### Langkah 1 - Melanjutkan alur project

Bagian ini menjalankan langkah berikutnya dalam proses kerja notebook. Bacalah bersama output di bawahnya, karena hasil dari cell ini biasanya menjadi jembatan menuju tahap berikutnya.

In [ ]:
# https://www.kaggle.com/datasets/spsayakpaul/arxiv-paper-abstracts/data# memprediksi kategori dari judul/abstrak

### Langkah 2 - Menyiapkan alat kerja

Kita mulai dengan memanggil library yang akan dipakai sepanjang notebook. Dengan menaruhnya di awal, pembaca bisa langsung melihat alat apa saja yang dibutuhkan sebelum project dijalankan.

In [ ]:
import typing as t
from ast import literal_eval

from transformer.models.classifier import ClassifierLM
from transformer.dataloaders.inference import InferenceDataModule
from transformer.params.transformer import TransformerParams

import torch
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from lightning import Trainer
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from transformers import LlamaTokenizer

### Langkah 3 - Memuat dan mengecek data awal

Di sini data dimuat lalu langsung ditampilkan beberapa baris pertamanya. Cara ini membantu pembaca memastikan file terbaca dengan benar sebelum masuk ke proses analisis.

In [ ]:
# memuat dan melihat pratinjau datadata = pd.read_csv("data/arxiv.csv")
data.titles = data.titles.str.replace("\n", " ")
data.abstracts = data.abstracts.str.replace("\n", " ")
data.tail()

### Langkah 4 - Melihat sebaran nilai

Bagian ini menghitung seberapa sering nilai tertentu muncul. Informasi ini berguna untuk melihat kelas yang dominan, data yang timpang, atau pola sederhana sebelum modeling.

In [ ]:
# mendapatkan gelar dan kategori utamaX = data.titles.to_list()
y = data.terms.apply(literal_eval).str[0]
print(y.value_counts())
y = y.to_numpy()

### Langkah 5 - Mengubah data kategori

Di sini nilai kategori diubah menjadi bentuk angka yang bisa dipahami model. Langkah ini dibutuhkan karena sebagian besar algoritma machine learning bekerja lebih baik dengan input numerik.

In [ ]:
# mengkodekan kategorilabel_encoder = LabelEncoder()
y = torch.from_numpy(label_encoder.fit_transform(y))

### Langkah 6 - Membuat fungsi bantu

Bagian ini membungkus proses tertentu ke dalam fungsi atau class agar kode berikutnya tidak berulang. Dengan cara ini, notebook menjadi lebih rapi dan alurnya lebih mudah dirawat.

In [ ]:
# membuat modul dataclass ArxivCategorizationDataModule(InferenceDataModule):
    def setup(self: t.Self, stage: str) -> None:
        self.X, self.y = X, y
        super().setup(stage=stage)

### Langkah 7 - Menyiapkan teks untuk model

Data teks perlu dibersihkan dan diubah menjadi bentuk numerik sebelum bisa dipelajari model. Langkah ini membantu kata-kata penting tetap terbaca sebagai pola, bukan sekadar kalimat mentah.

In [ ]:
# inisialisasi tokenizer yang telah dilatih sebelumnya# - llama tidak menambahkan token EOS secara default, jadi timpa ini# - llama juga tidak menggunakan token padding, jadi ini perlu ditambahkantokenizer = LlamaTokenizer.from_pretrained(
    "huggyllama/llama-7b", add_eos_token=True, legacy=False
)
tokenizer.add_special_tokens({"pad_token": "<pad>"})

### Langkah 8 - Mengecek variasi nilai

Di sini kita melihat jumlah atau daftar nilai unik pada kolom data. Langkah ini membantu memahami keragaman isi data, terutama untuk kolom kategori atau fitur yang nanti perlu diolah.

In [ ]:
# inisialisasi trafocontext_length = 64
model = ClassifierLM(
    params=TransformerParams(context_length=context_length),
    tokenizer=tokenizer,
    num_classes=len(y.unique())
)

### Langkah 9 - Menyiapkan teks untuk model

Data teks perlu dibersihkan dan diubah menjadi bentuk numerik sebelum bisa dipelajari model. Langkah ini membantu kata-kata penting tetap terbaca sebagai pola, bukan sekadar kalimat mentah.

In [ ]:
# memberi token & mengkodekan data dan menyiapkan pemisahan pelatihan/pengujiandatamodule = ArxivCategorizationDataModule(
    tokenizer=tokenizer,
    context_length=context_length,
    batch_size=32,
    val_size=0.2,
    test_size=0.1,
    num_workers=9,
    persistent_workers=True,
    limit=None,
    random_state=1,
)

### Langkah 10 - Melatih model

Bagian ini menjalankan proses training agar model belajar dari data yang sudah disiapkan. Output dari langkah ini biasanya menjadi petunjuk awal apakah model mulai menangkap pola dengan baik.

In [ ]:
%%time
# melatih modelnyatrainer = Trainer(
    max_epochs=50,
    callbacks=EarlyStopping(monitor="val_loss", mode="min", patience=5),
    accelerator="cpu",
)
trainer.fit(model=model, datamodule=datamodule)

### Langkah 11 - Melanjutkan alur project

Bagian ini menjalankan langkah berikutnya dalam proses kerja notebook. Bacalah bersama output di bawahnya, karena hasil dari cell ini biasanya menjadi jembatan menuju tahap berikutnya.

In [ ]:
# menghitung metrik pengujiantrainer.test(model=model, datamodule=datamodule)

### Langkah 12 - Membuat prediksi

Di sini model digunakan untuk menghasilkan prediksi dari data yang tersedia. Langkah ini menunjukkan bagaimana model dipakai setelah selesai dilatih, sehingga alurnya terasa lengkap dari data sampai hasil.

In [ ]:
# melihat kumpulan prediksi set pengujian pertamapred = trainer.predict(model=model, datamodule=datamodule)
pred[:10]

### Langkah 13 - Melanjutkan alur project

Bagian ini menjalankan langkah berikutnya dalam proses kerja notebook. Bacalah bersama output di bawahnya, karena hasil dari cell ini biasanya menjadi jembatan menuju tahap berikutnya.

In [ ]:
# menghitung akurasitorch.tensor([x[1] == x[2] for batch in pred for x in batch]).float().mean()